In [ ]:
import json, shutil, subprocess, sys, time
from pathlib import Path
RUN = 'base-s4'
OVERRIDES = {'seed': 20261031, 'steps': 16000}
code = next(p.parent for p in Path('/kaggle/input').rglob('train_jm.py'))
data = next(p.parent for p in Path('/kaggle/input').rglob('peaks.bin'))
jm = Path('/kaggle/working/jm'); (jm/'upstream').mkdir(parents=True, exist_ok=True)
for f in ['train_jm.py', 'model.py', 'features.py']: shutil.copy2(code/f, jm/f)
shutil.copy2(code/'upstream_train.py', jm/'upstream'/'train.py')
shutil.copy2(code/'features.py', jm/'upstream'/'features.py')
cfg = json.loads((code/'jm_base.json').read_text())
cfg.update(work=str(data), bank_dir=str(data), amp='fp16', hours=10.5, **OVERRIDES)
(jm/'run_config.json').write_text(json.dumps(cfg, indent=2))
out = Path('/kaggle/working')/RUN
print('code', code, 'data', data, 'config', cfg, flush=True)
subprocess.run(['nvidia-smi'])
t = time.time()
proc = subprocess.run([sys.executable, '-u', str(jm/'train_jm.py'), '--config', str(jm/'run_config.json'), '--output', str(out)])
print('exit', proc.returncode, 'minutes', round((time.time()-t)/60, 1), flush=True)
for f in ['last_model.pt', 'validation_predictions.csv']:
    (out/f).unlink(missing_ok=True)
print(sorted(p.name for p in out.iterdir()))
print((out/'training_result.json').read_text() if (out/'training_result.json').exists() else 'NO RESULT')
assert proc.returncode == 0